# Task 1 · SEC 10-K Sentiment Factor

**Research question:** does textual sentiment extracted from a company's 10-K annual report predict its next-month forward stock return on a cross-sectional basis?

**Method:** download 10-K filings for the S&P 100 universe (2018–2024), score each filing with the Loughran-McDonald finance-specific sentiment dictionary, build a monthly cross-sectional signal panel (each stock's most recent 10-K's `net_neg` score held forward for 12 months), and compute the Spearman IC vs forward 1-month returns.

**Factor:** `net_neg = (neg_count − pos_count) / n_tokens`. Per Loughran-McDonald 2011, the negative side of the LM list is the stronger predictor — their positive list is dominated by boilerplate.

**Benchmark comparison:** compare IC and cross-sectional correlation with three traditional factors — 12-month momentum, 1-month reversal, 12-month low-vol.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from src.data_fetcher import get_sp100_tickers, list_10k_paths, fetch_returns
from src.nlp_signals  import score_filings_from_manifest, enrich_filed_dates
from src.factor_analysis import (
    compute_ic_series, compute_ic_stats,
    quintile_returns, quintile_summary,
    build_signal_panel, factor_correlation_matrix,
    build_momentum_factor, build_lowvol_factor, build_mom1_reversal,
)

## 1 · Load universe and returns

In [ ]:
sp100 = get_sp100_tickers()
print(f'S&P 100: {len(sp100)} tickers, {sp100.sector.nunique()} sectors')
sp100.head()

In [ ]:
# Monthly CRSP returns via WRDS, yfinance fallback
rets = fetch_returns(sp100.ticker.tolist(), start='2017-01-01', end='2025-06-30')
forward_rets = rets.shift(-1)
print(f'returns panel: {rets.shape}')
rets.tail(3)

## 2 · Score all 10-K filings with LM dictionary

In [ ]:
# Already scored — load from parquet
scored = pd.read_parquet('../data/processed/filings_scored.parquet')
print(f'{len(scored)} filings scored, {scored.ticker.nunique()} tickers')
scored[['ticker','accession','filed_date','n_tokens','pos_count','neg_count','net_neg']].head()

In [ ]:
# Distribution of net_neg across all filings
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scored.net_neg * 10000, bins=40, color='steelblue', alpha=0.8)
ax.axvline(scored.net_neg.mean() * 10000, color='red', linestyle='--',
           label=f'mean = {scored.net_neg.mean()*10000:.0f} bp')
ax.set_xlabel('net_neg (basis points of total tokens)')
ax.set_ylabel('filing count')
ax.set_title('Distribution of Net Negativity across S&P 100 10-K filings (2018–2024)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 3 · Build monthly signal panel + IC analysis

The raw factor is `net_neg`. Since higher net negativity is hypothesized to predict **lower** returns, we use `-net_neg` as the directional signal (higher score = bullish).

In [ ]:
signal = -build_signal_panel(scored, 'net_neg', '2018-01-31', '2025-06-30', hold_months=12)
ic = compute_ic_series(signal, forward_rets)
ic_stats = compute_ic_stats(ic)
ic_stats

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ic.plot(ax=ax, alpha=0.6, linewidth=1, label='IC (monthly)')
ic.rolling(6).mean().plot(ax=ax, color='red', linewidth=2, label='6m rolling mean')
ax.axhline(0, color='k', linewidth=0.5)
ax.axhline(ic.mean(), color='green', linestyle='--', label=f'sample mean = {ic.mean():+.3f}')
ax.set_title(f'SEC 10-K Sentiment Factor — Spearman IC  (ICIR = {ic_stats["icir"]:+.2f})')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4 · Quintile portfolio returns

In [ ]:
qrets = quintile_returns(signal, forward_rets, n_bins=5, min_assets=20)
qsummary = quintile_summary(qrets)
pd.DataFrame(qsummary).T

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
(1 + qrets['LS']).cumprod().plot(ax=ax, color='navy', linewidth=1.5, label='L/S (Q5 − Q1)')
ax.axhline(1, color='k', linewidth=0.5)
ax.set_title('Long-Short Quintile Spread — Cumulative Return')
ax.set_ylabel('Growth of $1')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5 · Factor correlation vs traditional factors

In [ ]:
mom12 = build_momentum_factor(rets)
mom1  = build_mom1_reversal(rets)
lowvol = build_lowvol_factor(rets)

corr = factor_correlation_matrix({
    'sec_sentiment': signal,
    'mom12': mom12,
    'mom1_reversal': mom1,
    'lowvol': lowvol,
})
corr.round(3)

In [ ]:
# IC comparison table
factors = {'sec_sentiment': signal, 'mom12': mom12, 'mom1_reversal': mom1, 'lowvol': lowvol}
ic_comparison = pd.DataFrame({
    name: compute_ic_stats(compute_ic_series(panel, forward_rets))
    for name, panel in factors.items()
}).T
ic_comparison

## 6 · Findings

See `../README.md` and `../results/summary_stats.csv` for the written findings.